# MP27 — Self-Attention from Scratch

Learn how queries ask which keys match, how probabilities select values, and why dense attention needs quadratic token-pair memory. This is one untrained NumPy head, not a full Transformer or an SR result.

[Brief and commands](README.md) · [Validation](VALIDATION.md)

Prerequisites: matrix multiplication, softmax and MP26. All examples execute on CPU. No network calls or training. The recorded real example uses NAIP public-domain aerial RGB from the training split, normalized DN rather than calibrated reflectance.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import matplotlib.pyplot as plt
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
lesson=root/'stage_07_transformers/MP27_self_attention_from_scratch'
sys.path.insert(0,str(lesson))
from attention_tools import attention, softmax, score_bytes, entropy
np.set_printoptions(precision=6, suppress=True)


## 1. Shapes and a hand calculation

$Q\in\mathbb R^{N\times d}$ holds queries, $K\in\mathbb R^{M\times d}$ keys and $V\in\mathbb R^{M\times d_v}$ values. $A=\mathrm{softmax}_{row}(QK^T/\sqrt d)$ has shape $N\times M$; output $AV$ has shape $N\times d_v$. In self-attention N=M and all projections originate from the same token sequence. Feature values here are dimensionless numerical representations.

Take Q=[1,0], keys [1,0] and [0,1], values 10 and 20. Logits are [0.707107,0], so probabilities are approximately [0.669762,0.330238]. The weighted value is 13.302385. It is between the input values because the probabilities are nonnegative and sum to one.

In [ ]:
out, weights=attention([[1,0]],[[1,0],[0,1]],[[10],[20]])
p=np.exp(1/np.sqrt(2))/(np.exp(1/np.sqrt(2))+1)
np.testing.assert_allclose(weights,[[p,1-p]])
np.testing.assert_allclose(out,[[10*p+20*(1-p)]])
print('weights',weights,'output',out)


## 2. Stable row softmax

Subtract the largest logit before exponentiation. This does not change probabilities because the common exponential factor cancels. It avoids overflow for large finite logits; it cannot fix a matrix product that has already overflowed. The implementation rejects that input.

In [ ]:
np.testing.assert_allclose(softmax([[1000,1001]]),softmax([[0,1]]))
print(softmax([[1000,1001]]))


## 3. Masked attention and the zero-row error

True means an allowed key. False keys receive exactly zero probability. A row with no permitted key cannot define a probability distribution and raises ValueError. A diagonal mask makes each token select only its own value.

In [ ]:
values=np.array([[2.,4.],[7.,9.]])
out,a=attention(np.eye(2),np.eye(2),values,np.eye(2,dtype=bool))
np.testing.assert_array_equal(out,values)
try:
    softmax([[1,2]],np.zeros((1,2),dtype=bool))
except ValueError as error:
    print('Expected rejection:',error)
else:
    raise AssertionError('All-masked row must fail')


## 4. Feature scaling: predict, then inspect

Multiplying both Q and K by c multiplies the logits by c². Larger differences can make softmax nearly one-hot. Dividing by √d moderates feature-dimension variance under an independent unit-variance assumption; it does not cancel arbitrary feature rescaling.

We hold token count, values and seed fixed and compare scaled/unscaled attention at c=0.1,1,10,100. Entropy H=−Σp log(p), in nats, is log(64) for uniform rows and zero for one-hot rows. Entropy is not accuracy.

In [ ]:
record=json.loads((root/'reports/mp27-self-attention.json').read_text())
print(record['source'])
assert record['source']['split']=='train'
assert all(r['output_finite'] and r['max_row_sum_error']<1e-12 for r in record['rows'])
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,name in zip(axes,['synthetic_normal','naip_train_rgb']):
    for scaled in [False,True]:
        rows=[r for r in record['rows'] if r['input']==name and r['divide_by_sqrt_d']==scaled]
        ax.semilogx([r['feature_scale'] for r in rows],[r['mean_entropy_nats'] for r in rows],'o-',label='divide by sqrt(d)' if scaled else 'unscaled')
    ax.set(title=name,xlabel='Q and K feature multiplier',ylabel='Mean row entropy (nats)')
    ax.legend(); ax.grid(alpha=.2)
fig.tight_layout(); plt.show()


The recorded synthetic scaled entropy falls from about 4.159 to effectively zero as the multiplier grows. The real RGB crop is initially nearly uniform; at multiplier 100 the scaled entropy is 1.879 versus unscaled 0.709. Different feature distributions produce different saturation. No positional information, learned projections, SR quality or geographic generalization is demonstrated.

## 5. Memory growth without dangerous allocations

One float32 score matrix for h heads and N tokens needs 4hN² bytes. Doubling N multiplies storage by four. Doubling both image dimensions gives four times as many pixel tokens and sixteen times the score storage. These are analytical counts, not measured RAM. Training also stores gradients, activations and work buffers; the NumPy lesson uses float64.

In [ ]:
memory=record['memory']
assert score_bytes(256)==4*score_bytes(128)
fig,ax=plt.subplots(figsize=(7,4))
ax.loglog([r['tokens'] for r in memory],[r['float32_score_bytes']/2**20 for r in memory],'o-')
ax.set(xlabel='Token count N',ylabel='One float32 score matrix (MiB)',title='Analytical quadratic storage; one head')
ax.grid(alpha=.2); fig.tight_layout(); plt.show()
print('4096 tokens:',score_bytes(4096)/2**20,'MiB; no such matrix allocated')


## 6. Order and positional information

Without positional terms, permuting Q, K and V together permutes outputs in the same way. Attention alone does not know where a pixel was located. Test this property before adding positions in the next project.

In [ ]:
rng=np.random.default_rng(27); x=rng.normal(size=(5,3)); permutation=[2,0,4,1,3]
y,_=attention(x,x,x); yp,_=attention(x[permutation],x[permutation],x[permutation])
np.testing.assert_allclose(yp,y[permutation])
print('Permutation equivariance verified')


## Practice before viewing solutions

1. Change both values in the hand example to 7. Predict the output.
2. How many MiB does a 1024-token, four-head float32 score tensor need?
3. Explain why a uniform attention map is not evidence of poor SR.
4. Propose a fair test of √d scaling: what must remain fixed?

## Solutions and reflection

1. Output is 7: convex weights sum to one. 2. 16 MiB, excluding other allocations. 3. There is no trained reconstruction task or HR comparison here. 4. Hold Q/K/V samples, seeds and dimensions fixed when toggling the divisor; separately study dimensions with a declared random distribution.

Learned: shape contracts and stable normalization make attention inspectable. Failure: large features can saturate weights even with √d scaling. Next question: how do positional encodings and multiple heads change representations? These worked solutions are teaching evidence, not a claim that a learner has demonstrated mastery.

Reference: Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762), scaled dot-product attention. This project implements only that primitive.

In [ ]:
constant,_=attention([[1,0]],[[1,0],[0,1]],[[7],[7]])
np.testing.assert_allclose(constant,[[7]])
assert score_bytes(1024,heads=4)==16*2**20
print('Solution checks passed')
